# Feature Preparation for Li-CO2 and Li-O2 Models

This notebook contains four modules:

1. Prepare iteration-specific overpotential modeling features.
2. Split CO2 data using the fixed Li-air test compounds.
3. Split O2 data using the fixed Li-air test compounds.
4. Match precomputed composition and structural features to the split datasets.

Each module includes its own imports and can run independently
when its input files are available. Only Module 4 requires pymatgen.

Module 1 is configured for O2 iteration 0. Module 4 reads an existing
feature table; it does not generate structural predictions.

In [1]:
# ============================================================
# Optional: inspect the current Python environment
# ============================================================

import sys
import importlib.util

print("Python executable:", sys.executable)
print("pymatgen:", importlib.util.find_spec("pymatgen"))

Python executable: D:\app\miniconda\envs\p110p36\python.exe
pymatgen: None


In [2]:
# ============================================================
# Module 1. Prepare iteration-specific modeling features
# ============================================================

# ------------------------------------------------------------
# 1.1. Import dependencies
# ------------------------------------------------------------

import re
import json

import numpy as np
import pandas as pd

In [3]:
# ------------------------------------------------------------
# 1.2. Load compounds, structural parameters, and descriptors
# ------------------------------------------------------------

# Current configuration: O2 iteration 0.
# Select rows with index labels 0 through 33, inclusive.
tmcs = pd.read_excel(
    "data/0_O2.xlsx",
    sheet_name="TMCs",
    usecols=[
        "compounds",
        "a_1",
        "b_1",
        "c_1",
        "Grain_1",
        "a_2",
        "b_2",
        "c_2",
        "Grain_2",
        "CurrentDensity",
        "overpotential",
    ],
).loc[:33, :]

print("Iteration data shape:", tmcs.shape)

# Specify the element identifier and 13 metal descriptors.
cols_tm = [
    "element",
    "AN",
    "AW",
    "CR",
    "AR",
    "FIE",
    "SIE",
    "EA",
    "EN",
    "Nd",
    "Nv",
    "D",
    "EC",
    "TC",
]

# Preserve the original descriptor source.
# Replace missing values and "/" placeholders with zero.
tm = (
    pd.read_excel(
        "data/0_O2.xlsx",
        sheet_name="TM_descriptors",
        usecols=cols_tm,
    )
    .fillna(0, inplace=False)
    .replace("/", 0, inplace=False)
)

# Specify the element identifier and 12 nonmetal descriptors.
cols_c = [
    "element",
    "AN",
    "AW",
    "CR",
    "AR",
    "VR",
    "FIE",
    "SIE",
    "EA",
    "EN",
    "Np",
    "Nv",
    "D",
]

# Replace missing nonmetal descriptor values with zero.
c = (
    pd.read_excel(
        "data/0_O2.xlsx",
        sheet_name="C_descriptors",
        usecols=cols_c,
    )
    .fillna(0, inplace=False)
)


Iteration data shape: (27, 11)


In [4]:
# ------------------------------------------------------------
# 1.3. Define chemical formula parsing functions
# ------------------------------------------------------------

def parse_compound(compound):
    """Extract element symbols and coefficients from a simple formula.

    Parameters
    ----------
    compound : str
        Formula containing element symbols and optional integer
        or decimal stoichiometric coefficients.

    Returns
    -------
    dict
        Element symbols mapped to coefficients in their order
        of appearance.

    Notes
    -----
    An omitted coefficient is interpreted as 1.0.
    Parentheses are not supported.
    Repeated element symbols overwrite earlier values.
    """
    matches = re.findall(
        r"([A-Z][a-z]*)(\d*\.*\d*)",
        compound,
    )

    result = {}

    for match in matches:
        element, count = match

        # Use 1.0 when the formula omits an explicit coefficient.
        count = float(count) if count else 1.0
        result[element] = count

    return result


def get_res(compound_name):
    """Return the parsed elemental composition of a compound formula."""
    parsed_result = parse_compound(compound_name)
    return parsed_result


# Check parsing with a representative bimetallic nitride.
test = get_res("Mn0.2Cr0.8N")


In [5]:
# ------------------------------------------------------------
# 1.4. Inspect descriptors and the output schema
# ------------------------------------------------------------

display(tm.columns.tolist())
display(c.columns.tolist())

# Descriptor order follows the loaded descriptor tables.
columns = (
    [f"AB_{i}" for i in tm.columns.tolist()[1:]]
    + [f"C_{i}" for i in c.columns.tolist()[1:]]
    + [
        "a_1",
        "b_1",
        "c_1",
        "Grain_1",
        "a_2",
        "b_2",
        "c_2",
        "Grain_2",
        "CurrentDensity",
        "overpotential",
    ]
)

display(np.array(columns))

print(
    "Compound column data type:",
    tmcs["compounds"].dtype,
)


['element',
 'AN',
 'AW',
 'CR',
 'AR',
 'FIE',
 'SIE',
 'EA',
 'EN',
 'Nd',
 'Nv',
 'D',
 'EC',
 'TC']

['element',
 'AN',
 'AW',
 'CR',
 'AR',
 'VR',
 'FIE',
 'SIE',
 'EA',
 'EN',
 'Np',
 'Nv',
 'D']

array(['AB_AN', 'AB_AW', 'AB_CR', 'AB_AR', 'AB_FIE', 'AB_SIE', 'AB_EA',
       'AB_EN', 'AB_Nd', 'AB_Nv', 'AB_D', 'AB_EC', 'AB_TC', 'C_AN',
       'C_AW', 'C_CR', 'C_AR', 'C_VR', 'C_FIE', 'C_SIE', 'C_EA', 'C_EN',
       'C_Np', 'C_Nv', 'C_D', 'a_1', 'b_1', 'c_1', 'Grain_1', 'a_2',
       'b_2', 'c_2', 'Grain_2', 'CurrentDensity', 'overpotential'],
      dtype='<U14')

Compound column data type: object


In [6]:
# ------------------------------------------------------------
# 1.5. Construct composition-based features
# ------------------------------------------------------------

train_data = []

for index, compound in enumerate(tmcs["compounds"]):

    compound_feat = [compound]

    compound = get_res(compound)

    keys = list(compound.keys())
    val = list(compound.values())

    # Expected ternary formula order: metal A, metal B, nonmetal.
    if len(keys) > 2:

        A_key, A_index = keys[0], val[0]
        B_key, B_index = keys[1], val[1]

        # Weight elemental descriptors by stoichiometric coefficients.
        A_feat = (
            tm[tm["element"] == A_key]
            .values
            .flatten()[1:]
            * float(A_index)
        )

        B_feat = (
            tm[tm["element"] == B_key]
            .values
            .flatten()[1:]
            * float(B_index)
        )

        C_feat = (
            c[c["element"] == keys[2]]
            .values
            .flatten()[1:]
            * float(val[2])
        )

        # Sum the metal contributions without coefficient normalization.
        AB_feat = np.array([
            a + b
            for a, b in zip(A_feat, B_feat)
        ])

    else:

        # Expected binary formula order: metal, nonmetal.
        AB_key = keys[0]
        AB_index = val[0]

        AB_feat = (
            tm[tm["element"] == AB_key]
            .values
            .flatten()[1:]
            * float(AB_index)
        )

        C_feat = (
            c[c["element"] == keys[1]]
            .values
            .flatten()[1:]
            * float(val[1])
        )

    # Combine 25 composition descriptors, eight structural parameters,
    # and current density. Append the prediction target last.
    compound_feat += (
        list(AB_feat)
        + list(C_feat)
        + [
            tmcs.loc[index, "a_1"],
            tmcs.loc[index, "b_1"],
            tmcs.loc[index, "c_1"],
            tmcs.loc[index, "Grain_1"],
            tmcs.loc[index, "a_2"],
            tmcs.loc[index, "b_2"],
            tmcs.loc[index, "c_2"],
            tmcs.loc[index, "Grain_2"],
            tmcs.loc[index, "CurrentDensity"],

            # Preserve rounding of the target to two decimal places.
            round(tmcs.loc[index, "overpotential"], 2),
        ]
    )

    train_data.append(compound_feat)

In [7]:
# ------------------------------------------------------------
# 1.6. Assemble and export the iteration dataset
# ------------------------------------------------------------

columns = (
    [f"AB_{i}" for i in tm.columns.tolist()[1:]]
    + [f"C_{i}" for i in c.columns.tolist()[1:]]
    + [
        "a_1",
        "b_1",
        "c_1",
        "Grain_1",
        "a_2",
        "b_2",
        "c_2",
        "Grain_2",
        "CurrentDensity",
        "overpotential",
    ]
)

colnames = ["compound"] + columns

train_data_ = pd.DataFrame(
    train_data,
    columns=colnames,
)

train_data_.to_csv(
    "data/compound_0_O2.csv",
    index=False,
)

display(train_data_.columns)

Index(['compound', 'AB_AN', 'AB_AW', 'AB_CR', 'AB_AR', 'AB_FIE', 'AB_SIE',
       'AB_EA', 'AB_EN', 'AB_Nd', 'AB_Nv', 'AB_D', 'AB_EC', 'AB_TC', 'C_AN',
       'C_AW', 'C_CR', 'C_AR', 'C_VR', 'C_FIE', 'C_SIE', 'C_EA', 'C_EN',
       'C_Np', 'C_Nv', 'C_D', 'a_1', 'b_1', 'c_1', 'Grain_1', 'a_2', 'b_2',
       'c_2', 'Grain_2', 'CurrentDensity', 'overpotential'],
      dtype='object')

In [1]:
# ============================================================
# Module 2. Split CO2 data using the fixed Li-air test compounds
# ============================================================

import os
import pandas as pd

In [2]:
# ------------------------------------------------------------
# 2.1. Configure the output directory
# ------------------------------------------------------------

GROUP_COL = "compound"
SAVE_DIR = "data/end_to_end_split/co2"

os.makedirs(
    SAVE_DIR,
    exist_ok=True,
)


In [3]:
# ------------------------------------------------------------
# 2.2. Load the fixed Li-air compound split
# ------------------------------------------------------------

split_table = pd.read_csv(
    "data/end_to_end_split/outer_compound_split.csv"
)

split_table.columns = split_table.columns.str.strip()

split_table = split_table.rename(
    columns={"compounds": "compound"}
)

split_table[GROUP_COL] = (
    split_table[GROUP_COL]
    .astype(str)
    .str.strip()
)

air_test_compounds = set(
    split_table.loc[
        split_table["split"] == "test",
        GROUP_COL,
    ]
)


In [4]:
# ------------------------------------------------------------
# 2.3. Load and clean the final-iteration CO2 dataset
# ------------------------------------------------------------

co2_data = pd.read_csv(
    "data/compound_6_CO2.csv"
)

# Remove automatically generated CSV index columns.
co2_data = co2_data.loc[
    :,
    ~co2_data.columns.str.match(r"^Unnamed"),
].copy()

co2_data.columns = co2_data.columns.str.strip()

co2_data[GROUP_COL] = (
    co2_data[GROUP_COL]
    .astype(str)
    .str.strip()
)


In [5]:
# ------------------------------------------------------------
# 2.4. Create training and outer-test subsets
# ------------------------------------------------------------

# Match exact compound strings after trimming whitespace.
# Exclude all Li-air test compounds from the CO2 training pool.
co2_train_pool = (
    co2_data[
        ~co2_data[GROUP_COL].isin(
            air_test_compounds
        )
    ]
    .copy()
    .reset_index(drop=True)
)

co2_outer_test = (
    co2_data[
        co2_data[GROUP_COL].isin(
            air_test_compounds
        )
    ]
    .copy()
    .reset_index(drop=True)
)

# Report overlap in compound strings between the subsets.
overlap = (
    set(co2_train_pool[GROUP_COL])
    & set(co2_outer_test[GROUP_COL])
)

print("CO2 total rows:", len(co2_data))
print("CO2 total compounds:", co2_data[GROUP_COL].nunique())

print("CO2 training rows:", len(co2_train_pool))
print(
    "CO2 training compounds:",
    co2_train_pool[GROUP_COL].nunique(),
)

print("CO2 outer-test rows:", len(co2_outer_test))
print(
    "CO2 outer-test compounds:",
    co2_outer_test[GROUP_COL].nunique(),
)

print("Overlapping compounds:", len(overlap))

CO2 total rows: 708
CO2 total compounds: 122
CO2 training rows: 582
CO2 training compounds: 103
CO2 outer-test rows: 126
CO2 outer-test compounds: 19
Overlapping compounds: 0


In [6]:
# ------------------------------------------------------------
# 2.5. Save the split datasets
# ------------------------------------------------------------

co2_train_pool.to_csv(
    os.path.join(
        SAVE_DIR,
        "co2_train_pool.csv",
    ),
    index=False,
)

co2_outer_test.to_csv(
    os.path.join(
        SAVE_DIR,
        "co2_outer_test.csv",
    ),
    index=False,
)

In [1]:
# ============================================================
# Module 3. Split O2 data using the fixed Li-air test compounds
# ============================================================

import os
import pandas as pd

In [2]:
# ------------------------------------------------------------
# 3.1. Configure the output directory
# ------------------------------------------------------------

GROUP_COL = "compound"
SAVE_DIR = "data/end_to_end_split/o2"

os.makedirs(
    SAVE_DIR,
    exist_ok=True,
)

In [3]:
# ------------------------------------------------------------
# 3.2. Load the fixed Li-air compound split
# ------------------------------------------------------------

split_table = pd.read_csv(
    "data/end_to_end_split/outer_compound_split.csv"
)

split_table.columns = split_table.columns.str.strip()

split_table = split_table.rename(
    columns={"compounds": "compound"}
)

split_table[GROUP_COL] = (
    split_table[GROUP_COL]
    .astype(str)
    .str.strip()
)

air_test_compounds = set(
    split_table.loc[
        split_table["split"] == "test",
        GROUP_COL,
    ]
)


In [4]:
# ------------------------------------------------------------
# 3.3. Load and clean the final-iteration O2 dataset
# ------------------------------------------------------------

o2_data = pd.read_csv(
    "data/compound_6_O2.csv"
)

# Remove automatically generated CSV index columns.
o2_data = o2_data.loc[
    :,
    ~o2_data.columns.str.match(r"^Unnamed"),
].copy()

o2_data.columns = o2_data.columns.str.strip()

o2_data[GROUP_COL] = (
    o2_data[GROUP_COL]
    .astype(str)
    .str.strip()
)

In [5]:
# ------------------------------------------------------------
# 3.4. Create training and outer-test subsets
# ------------------------------------------------------------

# Match exact compound strings after trimming whitespace.
# Exclude all Li-air test compounds from the O2 training pool.
o2_train_pool = (
    o2_data[
        ~o2_data[GROUP_COL].isin(
            air_test_compounds
        )
    ]
    .copy()
    .reset_index(drop=True)
)

o2_outer_test = (
    o2_data[
        o2_data[GROUP_COL].isin(
            air_test_compounds
        )
    ]
    .copy()
    .reset_index(drop=True)
)

# Report overlap in compound strings between the subsets.
overlap = (
    set(o2_train_pool[GROUP_COL])
    & set(o2_outer_test[GROUP_COL])
)

print("O2 total rows:", len(o2_data))
print("O2 total compounds:", o2_data[GROUP_COL].nunique())

print("O2 training rows:", len(o2_train_pool))
print(
    "O2 training compounds:",
    o2_train_pool[GROUP_COL].nunique(),
)

print("O2 outer-test rows:", len(o2_outer_test))
print(
    "O2 outer-test compounds:",
    o2_outer_test[GROUP_COL].nunique(),
)

print("Overlapping compounds:", len(overlap))

O2 total rows: 443
O2 total compounds: 95
O2 training rows: 331
O2 training compounds: 75
O2 outer-test rows: 112
O2 outer-test compounds: 20
Overlapping compounds: 0


In [6]:
# ------------------------------------------------------------
# 3.5. Save the split datasets
# ------------------------------------------------------------

o2_train_pool.to_csv(
    os.path.join(
        SAVE_DIR,
        "o2_train_pool.csv",
    ),
    index=False,
)

o2_outer_test.to_csv(
    os.path.join(
        SAVE_DIR,
        "o2_outer_test.csv",
    ),
    index=False,
)

In [1]:
# ============================================================
# Module 4. Match precomputed composition and structural features
# ============================================================

import os
import pandas as pd

from pymatgen.core import Composition


In [2]:
# ------------------------------------------------------------
# 4.1. Configure input and output paths
# ------------------------------------------------------------

# This feature table must be generated separately.
structure_path = "data/compound_new_MCs_structure.csv"

dataset_files = {
    "co2_train": "data/end_to_end_split/co2/co2_train_pool.csv",
    "co2_test": "data/end_to_end_split/co2/co2_outer_test.csv",
    "o2_train": "data/end_to_end_split/o2/o2_train_pool.csv",
    "o2_test": "data/end_to_end_split/o2/o2_outer_test.csv",
}

output_dir = "data/end_to_end_model_data"

os.makedirs(
    output_dir,
    exist_ok=True,
)


In [3]:
# ------------------------------------------------------------
# 4.2. Define helpers for composition-based matching
# ------------------------------------------------------------

def find_compound_column(df):
    """Find the compound identifier column in a DataFrame.

    Parameters
    ----------
    df : pandas.DataFrame
        Table containing 'compound' or 'compounds'.

    Returns
    -------
    str
        Name of the compound identifier column.

    Raises
    ------
    KeyError
        If neither supported column name is present.
    """
    for column in ["compound", "compounds"]:

        if column in df.columns:
            return column

    raise KeyError(
        "No compound column found. "
        f"Available columns: {df.columns.tolist()}"
    )


def composition_key(formula):
    """Generate a scale-independent composition key for matching.

    Parameters
    ----------
    formula : str
        Chemical formula accepted by pymatgen Composition.

    Returns
    -------
    str
        Alphabetically ordered elemental fractions formatted to
        10 decimal places.

    Notes
    -----
    Element order and proportional formula scaling do not affect
    the key. This operation does not normalize model features.
    """
    comp = Composition(str(formula))

    amounts = comp.get_el_amt_dict()
    total = sum(amounts.values())

    normalized = {
        element: round(amount / total, 10)
        for element, amount in amounts.items()
    }

    return "|".join(
        f"{element}:{normalized[element]:.10f}"
        for element in sorted(normalized)
    )


In [4]:
# ------------------------------------------------------------
# 4.3. Load the precomputed feature table
# ------------------------------------------------------------

structure_df = pd.read_csv(
    structure_path
)

structure_compound_col = find_compound_column(
    structure_df
)

if structure_compound_col != "compound":

    structure_df = structure_df.rename(
        columns={
            structure_compound_col: "compound",
        }
    )

structure_df["composition_key"] = (
    structure_df["compound"]
    .apply(composition_key)
)


In [5]:
# ------------------------------------------------------------
# 4.4. Select numeric feature columns
# ------------------------------------------------------------

excluded_columns = {
    "compound",
    "compounds",
    "composition_key",
    "CurrentDensity",
    "overpotential",
    "label_probability",
}

# Select all numeric columns except those explicitly excluded.
# The selected columns may include both composition and structure features.
feature_columns = [
    column
    for column in structure_df.columns
    if column not in excluded_columns
    and pd.api.types.is_numeric_dtype(
        structure_df[column]
    )
]

print("Number of feature columns:", len(feature_columns))
print("Feature columns:")
print(feature_columns)

Number of feature columns: 33
Feature columns:
['AB_AN', 'AB_AW', 'AB_CR', 'AB_AR', 'AB_FIE', 'AB_SIE', 'AB_EA', 'AB_EN', 'AB_Nd', 'AB_Nv', 'AB_D', 'AB_EC', 'AB_TC', 'C_AN', 'C_AW', 'C_CR', 'C_AR', 'C_VR', 'C_FIE', 'C_SIE', 'C_EA', 'C_EN', 'C_Np', 'C_Nv', 'C_D', 'a_1', 'b_1', 'c_1', 'Grain_1', 'a_2', 'b_2', 'c_2', 'Grain_2']


In [6]:
# ------------------------------------------------------------
# 4.5. Deduplicate the source table by normalized composition
# ------------------------------------------------------------

duplicate_mask = structure_df[
    "composition_key"
].duplicated(keep=False)

if duplicate_mask.any():

    duplicate_count = structure_df.loc[
        duplicate_mask,
        "composition_key",
    ].nunique()

    print(
        "Duplicate composition keys:",
        duplicate_count,
    )

# Retain the first feature record for each normalized composition.
structure_features = (
    structure_df[
        ["composition_key"] + feature_columns
    ]
    .drop_duplicates(
        subset=["composition_key"],
        keep="first",
    )
    .reset_index(drop=True)
)

print(
    "Unique compositions with features:",
    len(structure_features),
)

Unique compositions with features: 25012


In [7]:
# ------------------------------------------------------------
# 4.6. Merge features into the CO2 and O2 datasets
# ------------------------------------------------------------

summary_records = []

for dataset_name, file_path in dataset_files.items():

    data = pd.read_csv(file_path)

    compound_col = find_compound_column(data)

    if compound_col != "compound":

        data = data.rename(
            columns={
                compound_col: "compound",
            }
        )

    data["composition_key"] = (
        data["compound"]
        .apply(composition_key)
    )

    # Replace existing columns that share names with source features.
    overlapping_features = [
        column
        for column in feature_columns
        if column in data.columns
    ]

    if overlapping_features:

        data = data.drop(
            columns=overlapping_features
        )

    # Multiple observations may share one composition-level feature row.
    merged = data.merge(
        structure_features,
        on="composition_key",
        how="left",
        validate="many_to_one",
        indicator=True,
    )

    # Identify compositions without a matching feature record.
    # This does not check for missing values within matched records.
    missing_mask = (
        merged["_merge"] == "left_only"
    )

    missing_compounds = (
        merged.loc[
            missing_mask,
            ["compound", "composition_key"],
        ]
        .drop_duplicates()
        .reset_index(drop=True)
    )

    print("\n" + "=" * 60)
    print(dataset_name)
    print("=" * 60)

    print("Rows:", len(merged))
    print(
        "Unique compounds:",
        merged["composition_key"].nunique(),
    )
    print(
        "Missing compounds:",
        len(missing_compounds),
    )

    if len(missing_compounds) > 0:

        display(missing_compounds)

        raise ValueError(
            f"Missing structural features in {dataset_name}"
        )

    # Remove temporary matching and merge-status columns.
    merged = merged.drop(
        columns=["_merge", "composition_key"]
    )

    # Arrange the compound identifier, source features, and current density.
    ordered_columns = ["compound"]

    ordered_columns += [
        column
        for column in feature_columns
        if column in merged.columns
    ]

    if "CurrentDensity" in merged.columns:
        ordered_columns.append("CurrentDensity")

    # Preserve remaining original columns.
    remaining_columns = [
        column
        for column in merged.columns
        if column not in ordered_columns
        and column != "overpotential"
    ]

    ordered_columns += remaining_columns

    # Keep the prediction target as the final column.
    if "overpotential" in merged.columns:
        ordered_columns.append("overpotential")

    merged = merged[ordered_columns]

    save_path = os.path.join(
        output_dir,
        f"{dataset_name}_with_structure.csv",
    )

    merged.to_csv(
        save_path,
        index=False,
    )

    # Count original compound strings in the exported summary.
    summary_records.append({
        "dataset": dataset_name,
        "rows": len(merged),
        "unique_compounds": merged["compound"].nunique(),
        "feature_columns": len(feature_columns),
        "missing_compounds": len(missing_compounds),
        "save_path": save_path,
    })

    print(f"[Saved] {save_path}")



co2_train
Rows: 582
Unique compounds: 103
Missing compounds: 0
[Saved] data/end_to_end_model_data\co2_train_with_structure.csv

co2_test
Rows: 126
Unique compounds: 19
Missing compounds: 0
[Saved] data/end_to_end_model_data\co2_test_with_structure.csv

o2_train
Rows: 331
Unique compounds: 75
Missing compounds: 0
[Saved] data/end_to_end_model_data\o2_train_with_structure.csv

o2_test
Rows: 112
Unique compounds: 20
Missing compounds: 0
[Saved] data/end_to_end_model_data\o2_test_with_structure.csv


In [8]:
# ------------------------------------------------------------
# 4.7. Export the preparation summary
# ------------------------------------------------------------

summary_df = pd.DataFrame(
    summary_records
)

summary_path = os.path.join(
    output_dir,
    "data_preparation_summary.csv",
)

summary_df.to_csv(
    summary_path,
    index=False,
)

print("\n" + "=" * 60)
print("Data preparation completed")
print("=" * 60)

display(summary_df)


Data preparation completed


,dataset,rows,unique_compounds,feature_columns,missing_compounds,save_path
0,co2_train,582,103,33,0,data/end_to_end_model_data\co2_train_with_stru...
1,co2_test,126,19,33,0,data/end_to_end_model_data\co2_test_with_struc...
2,o2_train,331,75,33,0,data/end_to_end_model_data\o2_train_with_struc...
3,o2_test,112,20,33,0,data/end_to_end_model_data\o2_test_with_struct...
